# Budget processing routine

This notebook contains code to pre-process online budgets into grouped budget terms that are then saved to file (to be then read in by, for example, `Mixed_Layer_Temperature_Budget.ipynb`).

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=4)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures')

# Load data

### Define paths, region to analyse and time period to analyse

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
output = 364 # 364 = 2017
#output = 365 # 365 = 2018
#output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors

tmp_folder = base + 'post_processed_diags/'

base2 = base + 'output%03d/ocean/' % output

# Subsample regions:
reg = [-270, -70, -60, 60] # Pacific

# Subsample time:
times = slice(None,None)
times_snap = slice(None,None) # Note; this must be 1 more than times.

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

### Load grid and standard variables

In [ ]:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)
rho0 = 1035.
Cp = 3992.10322329649

ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

### Load ml-binned budget variables and snapshots

In [ ]:
# Standard average daily budget diagnostics:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Falling average daily budget daignostics (while the name of the averaging is "risavg", in effect this is actually the falling average diagnostics):
ds_day_budget_falavg = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
ds_day_budget_falavg = ds_day_budget_falavg.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_falavg.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget.average_DT.data = ds_day_budget.average_DT*np.timedelta64(1,'D')
ds_day_budget_falavg.average_DT.data = ds_day_budget_falavg.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_day_budget = ds_day_budget.sel(time=times)
ds_day_budget_falavg = ds_day_budget_falavg.sel(time=times)

In [ ]:
# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

# Compute climatologies for selected standard variables:

In [ ]:
outputs = np.arange(336,366)                                           # Define outputs to include in climatology
fields = {'ocean_snapshot_month.nc':'all',                             # Define variables to compute
          'ocean_month.nc':['temp_in_mld','salt_in_mld','mld']
         }

dest_post = '_output%03d-%03d.clim.nc' % (outputs[0],outputs[-1])      # postfix for files

In [ ]:
for file in fields.keys():
    print('Doing ' + file + '...')
    ds = xr.open_dataset(base + 'output%03d' % outputs[0] + '/ocean/' + file,decode_times=False)
    if fields[file] != 'all':
        ds = ds[fields[file]]
    ds.load()

    for output in tqdm(outputs[1:]):
        ds2 = xr.open_dataset(base + 'output%03d' % output + '/ocean/' + file,decode_times=False)
        if fields[file] != 'all':
            ds2 = ds2[fields[file]]
        ds2.load()
        ds2 = ds2.assign_coords({'time':ds.time})
        ds = ds + ds2

    ds = ds/len(outputs)
    ds.to_netcdf(tmp_folder + file.replace('.nc',dest_post))        

In [ ]:
sreg = [150-360,168-360,-44,-38] # Tasman Sea region from Kajtar et al. 2022
region_name = 'Tasman Sea'
#sreg = [-70,5,0,60] # North Atlantic region
#region_name = 'North Atlantic'

## Plot time series averaged over the event region

Then, compute a daily-resolution climatology covering the period of interest

In [ ]:
add_extra = True # Whether to add a second year in the climatologies in order to cover the second half of December

year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])
year_clim = int(str(ds_clim.time[0].astype('datetime64[Y]').values)[:4])

# climatology, standard variables:
ds_climA = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in ds_clim.time]})
if add_extra:
    ds_clim2 = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in ds_clim.time]})
    ds_climA = xr.concat([ds_climA,ds_clim2],dim='time')
ds_climA = ds_climA.resample(time='1D').interpolate("linear").sel(time=times)

# Climatology, budget variables:
mlt_budget_stavg_climA = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in mlt_budget_stavg_clim.time]})
if add_extra:
    mlt_budget_stavg_clim2 = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in mlt_budget_stavg_clim.time]})
    mlt_budget_stavg_climA = xr.concat([mlt_budget_stavg_climA,mlt_budget_stavg_clim2],dim='time')
mlt_budget_stavg_climA = mlt_budget_stavg_climA#.resample(time='1D').interpolate("linear").sel(time=times)

Finally, plot the time series

In [ ]:
fig, axes = plt.subplots(nrows=4,ncols=1,figsize=(12,16),height_ratios=[1.,0.5,1.,1.])

# Panel 1: Mixed layer temperature, including climatology and snapshots:
mlt = (ds_day.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt_snap = (ds_day_snapshot.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt.plot(ax=axes[0],label='Daily-average mixed layer temperature',linewidth=5.)
mlt_snap.plot(ax=axes[0],label='Snapshot mixed layer temperature',linewidth=2.,linestyle='dashed')
(ds_climA.temp_in_mld/rho0).plot(ax=axes[0],label=clim_label + ' climatology',linewidth=2.)

# # Plot some averages:
# Octavg = mlt.sel(time=slice('2017-10-01','2017-10-31')).mean('time')
# Decavg = mlt.sel(time=slice('2017-12-01','2017-12-31')).mean('time')
# axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-11-01')],[Octavg.values,Octavg.values],'-',color='C0',linewidth=2.)
# axes[0].plot([np.datetime64('2017-12-01'),np.datetime64('2018-01-01')],[Decavg.values,Decavg.values],'-',color='C0',linewidth=2.)
# axes[0].plot([np.datetime64('2017-10-16T12:00:00'),np.datetime64('2017-12-16T12:00:00')],[Octavg.values,Decavg.values],'-',color='C0',linewidth=2.,linestyle='dashed',label='Epoch difference (Dec minus Oct)')
# axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-12-31')],[mlt_snap.sel(time='2017-10-01',method='nearest'),mlt_snap.sel(time='2017-12-31',method='nearest')],'-',color='C1',linewidth=2.,linestyle='dotted',label='Snapshot difference (24Z 31st Dec - 24Z 1st Oct)')

axes[0].legend()
axes[0].set_title(region_name + ' mixed layer temperature budget')
axes[0].set_ylabel('Temperature ($\circ$C)')
axes[0].grid()

# Panel 2: Mixed layer depth and climatology:
ds_day.mld.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).plot(ax=axes[1],linewidth=2.,label='Mixed layer depth')
(ds_climA.mld).plot(ax=axes[1],label=clim_label + ' climatology',linewidth=2.)
axes[1].set_ylabel('Mixed layer depth (m)')
axes[1].grid()
axes[1].legend()
axes[1].set_ylim([0.,150.])

# Panel 3: Budget terms (raw)
vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']

budget_stavg = mlt_budget_stavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
#budget_hatavg = mlt_budget_hatavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
budget_stavg_clim = mlt_budget_stavg_climA
unit_conv = 86400

for j, var in enumerate(vars):
    if j == 0:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j] + ' (st. avg.)')
#        (budget_hatavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,linestyle='dashed',label=labels[j] + ' (hat avg.)')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1.,label=labels[j] + ' (' + clim_label + ' climat.)')
    else:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j])
#        (budget_hatavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,linestyle='dashed')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1)
axes[2].legend()
axes[2].set_ylabel('Temperature tendency ($\circ$C/day)')
axes[2].grid()

# Panel 4: Budget terms (anomalies):

budget_stavg_clim_daily = mlt_budget_stavg_climA.interp(time=budget_stavg.time.astype('datetime64[s]'))
for j, var in enumerate(vars):
    ((budget_stavg[var]-budget_stavg_clim_daily[var])*unit_conv).plot(ax=axes[3],color='C' + str(j),linewidth=2,label=labels[j])
axes[3].legend()
axes[3].set_ylabel('Anomalous temperature \n tendency ($\circ$C/day)')
axes[3].grid()

for ax in axes:
    ax.set_xlabel('')
    ax.set_xlim([mlt.time[0],mlt.time[-1]])
    
#plt.savefig('MLT_budget_' + region_name.replace(' ','') + '_time_series_with_anomalies.png',dpi=250,bbox_inches='tight')

## Plot spatial plot of mixed layer temperature anomalies during event

Currently this is just for the Tasman Sea 2017 event

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(12,3.6),layout='constrained')

mlt = (ds_day.temp_in_mld.resample(time='1ME').mean()/rho0)
mlt_clim = (ds_clim.temp_in_mld/rho0)

(mlt.sel(time='2017-10').drop_vars(['time'])-mlt_clim.sel(time='1989-10').drop_vars(['time'])).plot(ax=axes[0],add_colorbar=False)
(mlt.sel(time='2017-11').drop_vars(['time'])-mlt_clim.sel(time='1989-11').drop_vars(['time'])).plot(ax=axes[1],add_colorbar=False)
(mlt.sel(time='2017-12').drop_vars(['time'])-mlt_clim.sel(time='1989-12').drop_vars(['time'])).plot(ax=axes[2],cbar_kwargs={'label': 'Mixed layer temperature \n anomaly ($^\circ$C)'})

axes[0].set_title('October 2017')
axes[1].set_title('November 2017')
axes[2].set_title('December 2017')
for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
axes[1].set_yticklabels([])
axes[2].set_yticklabels([])

#plt.savefig('MLT_budget_TasmanSea_OctDec2017_MLTA.png',dpi=300,bbox_inches='tight')

## Spatial plots of time-averaged budgets

This section plots spatial plots of the different contributions to budgets integrated over different time periods

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=6, figsize=(20,9))
axs = axes.reshape(-1)

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']
clim = 10

# Standard budget difference terms averaged over 3 month period (equivalent to snapshot difference):
times = slice('2017-10-01','2017-12-31')
stavg_budget = mlt_budget_stavg_daily_monthly.sel(time=times).sum('time')

for j, var in enumerate(vars):
    stavg_budget[var].where(stavg_budget[var]!=0.).plot(ax=axes[0][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[0][j].set_title('St. Avg ' + labels[j] + ' ($^\circ$C)')
axes[0][0].set_title(axes[0][0].get_title() + '\n (24Z 31st Dec - 24Z 1st Oct snapshot difference)')

# Hat average budget difference terms between 1st and last month (equivalent to time-average difference):
mlt_monthly = (ds_day.temp_in_mld/rho0).resample(time='1M').mean()
hatavg_budget = monthly_hat_difference(mlt_budget_risavg_monthly.sel(time=times),mlt_budget_stavg_monthly.sel(time=times),mlt_monthly.sel(time=times),0,len(mlt_budget_risavg_monthly.time.sel(time=times))-1)

for j, var in enumerate(vars):
    hatavg_budget[var].plot(ax=axes[1][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[1][j].set_title('Hat. Avg ' + labels[j] + ' ($^\circ$C)')
axes[1][0].set_title(axes[1][0].get_title() + '\n (Dec - Oct average difference)')

# Hat average budget difference terms between 1st and last day (equivalent to time-average difference) as a check:
mlt_daily = (ds_day.temp_in_mld/rho0)
hatavg_budget_daily = monthly_hat_difference(mlt_budget_risavg.sel(time=times),mlt_budget_stavg.sel(time=times),mlt_daily.sel(time=times),0,len(mlt_budget_risavg.time.sel(time=times))-1)

for j, var in enumerate(vars):
    hatavg_budget_daily[var].plot(ax=axes[2][j],cmap='RdBu_r',vmin=-clim,vmax=clim,extend='both',cbar_kwargs={'label':''}) 
    axes[2][j].set_title('Hat. Avg ' + labels[j] + ' ($^\circ$C)')
axes[2][0].set_title(axes[2][0].get_title() + '\n (31st Dec - 1st Oct average difference)')

for ax in axs:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
    
plt.tight_layout()
#plt.savefig('MLT_budget_TasmanSea_OctDec2017_spatial.png',dpi=250,bbox_inches='tight')

## Spatial plots of time-averaged online budget anomalies

Currently this is just for the North Atlantic 2023 event used in Matt's paper

In [ ]:
year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])
budget = mlt_budget_stavg_daily.resample(time='1ME').mean() # Take monthly mean
budget = budget.assign_coords({'time':budget.time.astype('datetime64[M]')}) # Replace time stamp with year-month only (to match climatology)
budget = budget - mlt_budget_stavg_clim.assign_coords({'time':[np.datetime64(str(year) + '-01')+np.timedelta64(x,'M') for x in range(12)]}) # Subtract climatology

In [ ]:
sreg = [-100, 20, 0, 60] # North Atlantic region
region_name = 'North Atlantic'
budget_av = (budget*ds_grid.area_t).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])/ds_grid.area_t.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12,6))
axs = axes.reshape(-1)

#times = ['2023-06','2023-07']
#time_labels = ['June 2023','July 2023']
times = ['2023-05','2023-06']
time_labels = ['May 2023','June 2023']
vars = [['mlt_tendency'],
        ['surface_flux','sw_pen'],
        ['advection','vert_mixing','entrainment']]
names = ['MLT tendency','Net surface flux term','Advection, mixing and entrainment']
clims = [-2.4,2.4]
unit_conv = 86400*30.5

for i,time in enumerate(times):
    budget_ti = budget.sel(time=np.datetime64(time,'M'))*unit_conv

    for j, varg in enumerate(vars):
        ds = budget_ti[varg[0]]
        if len(varg)>1:
            for k in range(len(varg)-1):
                ds += budget_ti[varg[k+1]]
        ds.plot.contourf(ax=axes[i][j],levels=np.arange(-2.4,2.8,0.4),cmap='RdBu_r')
        axes[i][j].set_title(names[j] + ' (' + time_labels[i] + ')')
        axes[i][j].set_xlabel('')
        axes[i][j].set_ylabel('')
        axes[i][j].set_facecolor([0.5,0.5,0.5])
        axes[i][j].set_xlim(reg[:2])
        axes[i][j].set_ylim([reg[2],reg[3]])

axes[0][0].plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
    
plt.tight_layout()
plt.savefig('MLT_budget_NorthAtlantic_spatial_May_Jun.png',dpi=200,bbox_inches='tight')

## Bar plots of region averages:

Again, just for the North Atlantic case currently

In [ ]:
fig = plt.figure(figsize=(12,15))
ax = plt.gca()

times = ['2023-05','2023-06','2023-07','2023-08']
time_labels = ['May','June','July','August']
vars = [['mlt_tendency'],
        ['surface_flux','sw_pen'],
        ['shortwave','sw_pen'],
        ['latent'],
        ['longwave'],
        ['sensible'],
        ['vert_mixing','entrainment'],
       ['advection']]
names = ['MLT tendency','Net surface flux','Shortwave','Latent','Longwave','Sensible','Vertical mixing and entrainment','Advection']

# Create variable groups:
budget_av_gr = budget_av['mlt_tendency'].rename(names[0]).to_dataset()
for i, varg in enumerate(vars[1:]):
    budget_av_gr[names[i+1]] = budget_av[varg[0]]
    if len(varg)>1:
        for k in range(len(varg)-1):
            budget_av_gr[names[i+1]] += budget_av[varg[k+1]]
    
df = budget_av_gr.rename({'time':'class'}).assign_coords({'class':time_labels}).to_dataframe().reset_index()
   
# Parameters
num_vars = len(times)
num_classes = len(names)
bar_width = 0.1
x = np.arange(num_vars)  # One x position per variable

unit_conv = 86400*30.5
# Create figure

# Plot each class as a separate bar group
for i, cls in enumerate(names):
    # Get values for this class across all variables
    values = list(df[cls]*unit_conv)
    
    # Offset x positions for each class
    ax.bar(x + i * bar_width, values, width=bar_width, label=cls)

# Formatting
ax.set_xticks(x + bar_width)
ax.set_xticklabels(time_labels)
ax.set_yticks(np.arange(-0.8,3.2,0.4))
ax.set_ylim([-0.8,1.8])
ax.set_ylabel("$^\circ$C/month")
ax.set_title('North Atlantic MLT budget anomalies 2023')
ax.legend(title="Budget terms",fontsize=8,loc='upper right',bbox_to_anchor=[1.1,0.9,0.1,0.1])
ax.grid()
plt.tight_layout()
plt.savefig('MLT_budget_NorthAtlantic_time_series_big.png',dpi=200,bbox_inches='tight')

In [ ]:
budget_av.surface_flux.values

In [ ]:
budget_av.sw_pen.values

In [ ]:
budget_av.mlt_tendency.values

# Define functions to construct grouped Mixed-layer temperature budget terms

First we define the term groups

In [ ]:
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}

Then define functions to compute the correction terms, do the grouping and compute the tendency and entrainment terms

In [ ]:
def compute_corrections(ds_day_budget):
    """
    Compute corrections to advection, surface mass flux and total tendency terms in ds_day_budget.

    For the P-E+R correction:
    -------------------------
    
    sfc_hflux_pme_in_mld              =     Ca*Qm/H*Cp                      (Wm-3)
    pme_river_times_temp_in_mld       =     CH*Qm/H                         (kg m-3 deg C s-1)
    sfc_hflux_pme_in_mld_cor          =     (Ca-CH)*Qm/H*Cp                 (Wm-3)

    For the advection correction:
    -----------------------------
    adv_cor1                          =     CH*divU/H                       (deg C s-1), where divU = Qm/rho0 + Ssmoother - deta/dt from the free-surface equation
    adv_cor2                          =     Cent*(1- H/(D+eta))*deta/dt/H   (deg C s-1)

    Produces the corrected terms sfc_hflux_pme_in_mld_cor, temp_eta_smooth_in_mld_cor, temp_advection_in_mld_cor and temp_tendency_in_mld_cor
    """

    # P-E+R correction: 
    pme_cor = -ds_day_budget['pme_river_times_temp_in_mld']/rho0
    ds_day_budget['sfc_hflux_pme_in_mld_cor'] = ds_day_budget['sfc_hflux_pme_in_mld'] + pme_cor*rho0*Cp

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_temp_in_mld']/rho0
    ds_day_budget['temp_eta_smooth_in_mld_cor'] = ds_day_budget['temp_eta_smooth_in_mld'] + eta_smoother_cor*rho0*Cp

    # Advection correction:
    adv_cor1 = (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0
    adv_cor2 = ds_day_budget['s_surf_ent_temp']/rho0
    ds_day_budget['temp_advection_in_mld_cor'] = ds_day_budget['temp_advection_in_mld'] + adv_cor1*rho0*Cp  + adv_cor2*rho0*Cp

    # Total correction for entrainment-by-residual:
    ds_day_budget['temp_tendency_in_mld_cor'] = ds_day_budget['temp_tendency_in_mld']  + pme_cor*rho0*Cp + adv_cor1*rho0*Cp + adv_cor2*rho0*Cp + eta_smoother_cor*rho0*Cp

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    if do_extras=True, also add extra budget terms (e.g. the components of the surface flux) from bud_var_extras.
    """
    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget[bud_tendency]/rho0/Cp).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0/Cp
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0/Cp
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0/Cp

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

# Define functions to construct grouped Mixed-layer salinity budget terms

As above, but for salinity

In [ ]:
bud_tendency = 'salt_tendency_in_mld_cor'
bud_var_grps = {'advection':['salt_advection_in_mld_cor',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',       # This is the restoring term - if wanting to separate it out.
                                'pme_in_mld_cor',                 # THis is the surface freshwater flux term - the main one. It enters as a "correction" term in the language of the MLT budget described in the theory above and in the paper.
                                'salt_eta_smooth_in_mld_cor']}
bud_var_extras = {}

In [ ]:
def compute_corrections(ds_day_budget):
    """
    Compute corrections to advection and surface mass flux terms in ds_day_budget for salinity
    """

    # P-E+R correction:
    # pme_river_times_salt_in_mld = psu kg m -3 s-1
    pme_cor = -ds_day_budget['pme_river_times_salt_in_mld']/1000.
    ds_day_budget['pme_in_mld_cor'] = pme_cor

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_salt_in_mld']/1000.
    ds_day_budget['salt_eta_smooth_in_mld_cor'] = ds_day_budget['salt_eta_smooth_in_mld'] + eta_smoother_cor

    # Advection correction:
    adv_cor1 = -(ds_day_budget['eta_t_tendency_times_salt_in_mld']-ds_day_budget['pme_river_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.
    adv_cor2 = ds_day_budget['s_surf_ent_salt']/1000.
    ds_day_budget['salt_advection_in_mld_cor'] = ds_day_budget['salt_advection_in_mld'] + adv_cor1 + adv_cor2

    # Total correction for residual:
    ds_day_budget['salt_tendency_in_mld_cor'] = ds_day_budget['salt_tendency_in_mld']  + pme_cor + adv_cor1 + adv_cor2 + eta_smoother_cor

    return(ds_day_budget)

def mlt_budget_fixedh(ds_day_budget, do_extras = True):
    """
    Compute fixedh budget terms, including residual. Does not include entrainment or mlt tendency.
    if do_extras=True, also add extra budget terms (e.g. the components of the surface flux) from bud_var_extras.
    """
    # Extract variable sums in groups, dividing by rho0*Cp to convert to degC/second
    mlt_budget = (ds_day_budget[bud_tendency]/rho0*1000.).rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        mlt_budget[var] = ds_day_budget[bud_var_grps[var][0]]/rho0*1000.
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                mlt_budget[var] += ds_day_budget[raw_var]/rho0*1000.
    
    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        mlt_budget['residual'] -= mlt_budget[var]

    if do_extras:
        for var in bud_var_extras.keys():
            mlt_budget[var] = ds_day_budget[bud_var_extras[var][0]]/rho0*1000.
            if (len(bud_var_extras[var])>1):
                for raw_var in bud_var_extras[var][1:]:
                    mlt_budget[var] += ds_day_budget[raw_var]/rho0*1000.

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

# Compute budget terms

Note: These cells are just for testing, if you're working with pre-computed budget terms, none of this code should be needed.

## daily budget, standard averaging

In [ ]:
# Compute in one go:
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily = compute_tendency_entrainment(mlt_budget_stavg_daily,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily.load();

In [ ]:
# Compute in blocks (e.g. if doing a whole year):

ds_day_budget = compute_corrections(ds_day_budget)

# NOTE: THIS still seems way slower than it should be... Something simple might make it faster...
bs = 30; tl = len(ds_day_budget.time)
blocks = [range(tl)[x*bs:(x+1)*bs] for x in range(int(np.ceil(tl/bs)))]
blocks_snap = [range(tl+1)[x*bs:(x+1)*bs + 1] for x in range(int(np.ceil(tl/bs)))]

mlt_budget_stavg_daily_uncat = []
for i in tqdm(range(len(blocks))):
    bud = mlt_budget_fixedh(ds_day_budget.isel(time=blocks[i]))
    bud = compute_tendency_entrainment(bud,ds_day_snapshot.temp_in_mld.isel(time=blocks_snap[i])/rho0)
    mlt_budget_stavg_daily_uncat.append(bud.load())
mlt_budget_stavg_daily = xr.concat(mlt_budget_stavg_daily_uncat,dim='time')

In [ ]:
# Save to file if desired:
mlt_budget_stavg_daily.to_netcdf(tmp_folder + 'mlt_budget_stavg_daily_online_output%03d.nc' % output)

## Compute monthly budget climatology

Note: There is currently some bug with the monthly budget's accumulations - temp_tendency_in_mld shows a signature of the daily cycle (zonal cooling in the east pacific and warming zonally elsewhere). I don't know what it is. The daily files seem fine.

### From raw climatology files (e.g. climatology computed before term grouping/entrainment/tendency etc.)

In [ ]:
# Compute fixedh budget:
mlt_budget_stavg_clim = mlt_budget_fixedh(ds_clim_budget)

# Compute entrainment and tendnecy:
mlt_budget_stavg_clim = compute_tendency_entrainment(mlt_budget_stavg_clim,ds_clim_snapshot.temp_in_mld/rho0)

# Force computation:
mlt_budget_stavg_clim.load();

### From pre-computed climatology files (e.g. including pre-computed grouping/entrainment/tendency etc.)

In [ ]:
base = '/g/data/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'
outputs = np.arange(336,366) # 1989-2018
dest_folder = base + 'clim_1989-2018/'

for output in tqdm(outputs):

    ######################## Load data for this year:

    base2 = base + 'output%03d' % output + '/ocean/'
    
    # Standard average daily budget diagnostics:
    ds_mon_budget = xr.open_dataset(base2 + 'ocean_budget_month.nc',decode_times=False,chunks=chunks2D)
    
    # Snapshots for standard average tendency computation:
    ds_mon_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D)
    # Add previous output for last element:
    ds_mon_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D)
    ds_mon_snapshot = xr.concat([ds_mon_snapshot_m1.isel(time=-1),ds_mon_snapshot],dim='time')
    
    # Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
    ds_mon_budget = ds_mon_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget.time.values]})
    ds_mon_snapshot = ds_mon_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_snapshot.time.values]})
    
    # Fix average_DT by decoding by hand:
    ds_mon_budget.average_DT.data = ds_mon_budget.average_DT*np.timedelta64(1,'D')
    
    # Subselect time period:
    ds_mon_budget = ds_mon_budget.sel(time=times)
    ds_mon_snapshot = ds_mon_snapshot.sel(time=times_snap)

    ############### Compute budget:
    mlt_budget_stavg_monthly = mlt_budget_fixedh(ds_mon_budget)
    mlt_budget_stavg_monthly = compute_tendency_entrainment(mlt_budget_stavg_monthly,ds_mon_snapshot.temp_in_mld/rho0)
    mlt_budget_stavg_monthly.load();

    ############### Save to file:
    mlt_budget_stavg_monthly.to_netcdf(tmp_folder + 'mlt_budget_stavg_monthly_online_output%03d.nc' % output)
    

In [ ]:
mlt_budget_stavg_monthly.isel(time=0).entrainment.plot()

## Compute monthly difference budget, standard averaging
`mlt_budget_stavg_monthly` is simply the time integral of of the daily `mlt_budget_stavg_daily` budget. The resulting array contains the temperature difference induced by each term (or the temperature difference itself, for mlt\_tendency) across the month. Units are degC.

In [ ]:
mlt_budget_stavg_monthly = (mlt_budget_stavg_daily*(ds_day.average_DT/np.timedelta64(1,'s'))).resample(time='1ME').sum()
mlt_budget_stavg_monthly['time'] = mlt_budget_stavg_daily.time.resample(time='1ME').mean() # Centre time in the middle of the month.

In [ ]:
ds = mlt_budget_stavg_monthly.entrainment.load()

In [ ]:
# Another res check:
ds = mlt_budget_stavg_daily['mlt_tendency'] - mlt_budget_stavg_daily['advection'] - mlt_budget_stavg_daily['surface_flux']  - mlt_budget_stavg_daily['vert_mixing']   - mlt_budget_stavg_daily['sw_pen']    - mlt_budget_stavg_daily['entrainment']

In [ ]:
(ds.isel(time=-10)*86400).plot()

In [ ]:
fig, axes  = plt.subplots(nrows=3,ncols=4,figsize=(20,10))
for i, var in enumerate(mlt_budget_stavg_daily.data_vars):
    (mlt_budget_stavg_daily[var]*86400).isel(time=-10).plot(ax=axes.reshape(-1)[i],vmin=-.5,vmax=.5,cmap='RdBu_r')
    axes.reshape(-1)[i].set_title(var)
    #ds.isel(time=mn).plot(ax=axes.reshape(-1)[mn],vmin=-2.,vmax=2.,cmap='RdBu_r')
plt.tight_layout()

## Compute daily budget, hat averaging

Compute the hat-averaged daily budget. This budget (containing the same fields as `mlt_budget_stavg_daily`) corresponds to the tendency of the *daily-averaged* mixed layer temperature. E.g., mlt\_tendency in `mlt_budget_hatavg_daily` is the difference in daily-averaged temperature between the day after and the day before the time stamp (with the time stamp being at 0Z inbetween the two days), divided by the number of seconds in the day (units degC/second). The other terms are the budget contributions to this tendency. 

In [ ]:
def hat_average(st_avg,fal_avg_raw,average_DT):
    """
    Compute hat average from standard and falling average for a particular field
    """
    fal_avg = fal_avg_raw/(average_DT/np.timedelta64(1,'s')) # This line fixes a bug in the normalization of the daily falling average diagnostics
                                                             # take mean by dividing by averaging period. We do this here to avoid loading all the variables just to do this fix.
    ris_avg = st_avg - fal_avg                               # Rising = standard - falling
    hat_avg = ris_avg.isel(time=slice(0,-1)).values +  fal_avg.isel(time=slice(1,None)) # Hat = rising over first day - falling over second day              
                                                                                        # Note: Dealing with time is done outside this function
    return(hat_avg)

In [ ]:
# Template variable (note that hat average tendencies lie on snapshot (1:end-1) time:
mlt_budget_hatavg_daily = np.nan*xr.zeros_like(ds_day_snapshot.temp_in_mld.isel(time=slice(1,-1)).transpose(*ds_day_budget['temp_tendency_in_mld'].dims)) 

# Compute hat average fixedh tendency:
mlt_budget_hatavg_daily.data = hat_average(ds_day_budget['temp_tendency_in_mld'],ds_day_budget_falavg['temp_tendency_in_mld'],ds_day_budget_falavg.average_DT)/rho0/Cp

# Make a dataset:
mlt_budget_hatavg_daily = mlt_budget_hatavg_daily.rename('fixedh_tendency').to_dataset()

# Do other variables:
for var in bud_var_grps.keys():
    mlt_budget_hatavg_daily[var] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
    mlt_budget_hatavg_daily[var].data = hat_average(ds_day_budget[bud_var_grps[var][0]],ds_day_budget_falavg[bud_var_grps[var][0]],ds_day_budget_falavg.average_DT)/rho0/Cp
    if (len(bud_var_grps[var])>1):
        for raw_var in bud_var_grps[var][1:]:
            mlt_budget_hatavg_daily[var].data += hat_average(ds_day_budget[raw_var],ds_day_budget_falavg[raw_var],ds_day_budget_falavg.average_DT)/rho0/Cp

# Compute residual for check:
mlt_budget_hatavg_daily['residual'] = mlt_budget_hatavg_daily['fixedh_tendency'].copy(deep=True)
for var in list(mlt_budget_hatavg_daily.data_vars):
    mlt_budget_hatavg_daily['residual'] -= mlt_budget_hatavg_daily[var]

# Compute mlt tendency:
mlt = (ds_day.temp_in_mld/rho0).transpose(*mlt_budget_hatavg_daily['fixedh_tendency'].dims)
mlt_budget_hatavg_daily['mlt_tendency'] = xr.zeros_like(mlt_budget_hatavg_daily['fixedh_tendency']).copy(deep=True)
mlt_budget_hatavg_daily['mlt_tendency'].data = mlt.isel(time=slice(1,None)).values - mlt.isel(time=slice(0,-1)).values
mlt_budget_hatavg_daily['mlt_tendency'] = mlt_budget_hatavg_daily['mlt_tendency']/86400. # Note: this will only work for daily averaging, 
                                                                                           # as it assumes a 86400 time difference between 
                                                                                           # the centre of the two time-averaged periods (day before to day after)

# Entrainment term by residual:
mlt_budget_hatavg_daily['entrainment'] = -(mlt_budget_hatavg_daily['fixedh_tendency'] - mlt_budget_hatavg_daily['mlt_tendency'])

## Compute monthly difference budget, hat averaging

This section computes the monthly difference budgets (e.g. as above, contributions to the temperature differences across individual months) for standard averaging and hat averaging from the daily budgets. It then defines a function `monthly\_hat\_average` that takes these budgets as inputs, along with the monthly-averaged mixed layer temperature and two month indexes of interest, and outputs the contributions to the budget that govern the difference between the monthly-averaged mixed layer temperature of those two months.

Note: Some code and calculations here are repeated from the daily averaging performed above for conveninence.

In [ ]:
# Daily standard average budget (duplicated from above):
mlt_budget_stavg = mlt_budget_fixedh(ds_day_budget)

# Multiply by Delta t for differences:
mlt_budget_stavg = mlt_budget_stavg*(ds_day_budget.average_DT/np.timedelta64(1,'s'))

In [ ]:
# Daily falling average budget:
mlt_budget_falavg = mlt_budget_fixedh(ds_day_budget_falavg)

In [ ]:
# Daily rising average:
mlt_budget_risavg = mlt_budget_stavg - mlt_budget_falavg

In [ ]:
# Define n-1 DataArrays for the months:
n_minus_1 = xr.DataArray(data=[x-1 for x in mlt_budget_risavg.time.dt.day.values],dims=['time'],coords={'time':mlt_budget_stavg.time})

# Monthly rising average:
mlt_budget_risavg_monthly = mlt_budget_risavg.resample(time='1ME').mean() + (mlt_budget_stavg*n_minus_1).resample(time='1ME').mean()

# Monthly standard average:
mlt_budget_stavg_monthly = mlt_budget_stavg.resample(time='1ME').sum()

In [ ]:
# Define function to compute full budget given two months of interest:
def monthly_hat_difference(mlt_budget_risavg_monthly,mlt_budget_stavg_monthly,mlt_monthly,month1_index,month2_index):
    """
    Compute hat difference mlt budget from month 1 to month 2. 

    Inputs:
    - mlt_budget_risavg_monthly: The monthly difference budget, rising average
    - mlt_budget_stavg_monthly: The monthly difference budget, standard average
    - mlt_monthly: The monthly-average mixed layer temperature
    - month1_index: The index of the first month
    - month2_index: The index of the second month
    """

    # List of variables:
    vars = list(mlt_budget_risavg_monthly.data_vars)

    # Interim months:
    monthM_index = np.arange(month1_index+1,month2_index,1)
    
    # Compute mlt difference as "mlt_tendency":
    mlt_budget_hat_diff = (mlt_monthly.isel(time=month2_index) - mlt_monthly.isel(time=month1_index)).rename('mlt_tendency').to_dataset()

    # Compute falling difference as difference between other budgets:
    mlt_budget_falavg_monthly = mlt_budget_stavg_monthly - mlt_budget_risavg_monthly

    # Compute budget terms:
    for var in vars:
        mlt_budget_hat_diff[var] = mlt_budget_risavg_monthly[var].isel(time=month1_index) + mlt_budget_stavg[var].isel(time=monthM_index).sum('time') + mlt_budget_falavg_monthly[var].isel(time=month2_index)

    # Compute entrainment by residual:
    mlt_budget_hat_diff['entrainment'] = -(mlt_budget_hat_diff['fixedh_tendency'] - mlt_budget_hat_diff['mlt_tendency'])

    return(mlt_budget_hat_diff)